## Settings

In [1]:
## auto reload modules
%reload_ext autoreload
%autoreload 2

## Dependencies

In [2]:
## libraries
import sys
import pickle
import pandas as pd
from pathlib import Path
from itertools import combinations

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.data.builders import load_processed_data
from src.estimators.factories.registry import load_estimators
from src.evaluators.caching import load_notebook_cache
from src.evaluators.training import fit_predict_frontier
from src.evaluators.metrics import frontier_consensus

## constants
from src.evaluators.config import CONSENSUS_METRICS
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z,
    TARGET,
)

## Initialization

In [3]:
## reproducibility
N_DECIMALS = 2
N_REPEATS = 30
RANDOM_STATE = 42
FORCE_RECOMPUTE = False

## load data and models
data = load_processed_data()
models = load_estimators(random_state = RANDOM_STATE)

## view data shape
n_obs, n_feat = data.shape
print(f"Original Data: {n_feat} features, {n_obs} observations")

## view model surface
n_mods = len(models)
print(f"Learned Models: {n_mods} estimators")

## cache setup
cache_path = root / "notebooks" / "cache" / "consensus_results.pkl"
cache_metadata = {
    "n_obs": len(data),
    "n_repeats": N_REPEATS,
    "random_state": RANDOM_STATE,
    "model_names": sorted(models.keys()),
    "target": TARGET,
    "feat_x": list(FEAT_X),
    "feat_z": list(FEAT_Z),
}
cache_keys = ("frontiers", "results_data")
cache_payload = load_notebook_cache(
    cache_path = cache_path,
    metadata = cache_metadata,
    required_keys = cache_keys,
    force_recompute = FORCE_RECOMPUTE,
)
if cache_payload is not None:
    globals().update({key: cache_payload[key] for key in cache_keys})

Original Data: 32 features, 25 observations
Learned Models: 9 estimators
Loaded results from /Users/nickkunz/Documents/Nick’s MacBook Pro/Python/network-capacity/notebooks/cache/consensus_results.pkl


## Training

In [4]:
## train models
if cache_payload is None:
    frontiers = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        fit_result = fit_predict_frontier(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            n_repeat = N_REPEATS,
            random_state = RANDOM_STATE,
        )
        frontiers[name] = fit_result["y_pred"]

    ## compute pairwise consensus
    results_list = list()
    for (name_true, y_true), (name_pred, y_pred) in combinations(iterable = frontiers.items(), r = 2):

        ## calculate all metrics
        metrics = frontier_consensus(y_true = y_true, y_pred = y_pred)
        results_list.append({
            "model_i": name_true,
            "model_j": name_pred,
            **metrics,
        })

## Post-Processing

In [5]:
if cache_payload is None:
    ## convert to dataframe
    results_data = pd.DataFrame(results_list)

    ## persist canonical consensus results for visualization notebooks
    cache_path.parent.mkdir(parents = True, exist_ok = True)
    cache_payload = {
        "metadata": cache_metadata,
        "frontiers": frontiers,
        "results_data": results_data,
    }
    cache_path.write_bytes(pickle.dumps(cache_payload))
    print(f"Cached consensus results -> {cache_path}")

## Model Consensus Evaluation
Pairwise agreement between fitted frontiers evaluates whether independently trained estimators converge on aligned capacity orderings. Each learner is fit 30 times on the full dataset, and predictions are averaged across repeats before agreement is computed across all model pairs.

- **Spearman (`ρ`)**: Global rank agreement between paired model frontiers.
- **Rank-Biased Overlap (`RBO`)**: Top-weighted agreement, emphasizing the high-capacity tail.
- **Distance Correlation (`dCor`)**: Nonlinear dependence between paired model frontiers.
- **Consensus Index (`CI`)**: Geometric mean of agreement-scaled `ρ`, `RBO`, and `dCor`.

### Reporting Convention
Each model pair contributes one value per metric. The summary reports medians across all pairwise model comparisons. Prior to compounding, `ρ` is rescaled from `[-1, 1]` to `[0, 1]` so that negative rank agreement lowers `CI` without collapsing to a single floor. Pairwise comparisons are equally weighted. Higher values indicate stronger cross-model consensus.


In [6]:
## preview results
round(number = results_data[CONSENSUS_METRICS].median(), ndigits = N_DECIMALS)

rho    0.95
rbo    0.86
dcr    0.91
ci     0.90
dtype: float64

_Pairwise estimator comparisons show consistently high cross-model consensus. Median global rank agreement reaches 0.89 (ρ) and nonlinear dependence alignment reaches 0.91 (dCor), with a Consensus Index (CI) of 0.90 reflecting strong agreement across all dimensions simultaneously. Top-end overlap is also high at 0.86 (RBO), indicating that alignment extends to the high-capacity tail rather than being driven by mid-range agreement alone. Across methodologically distinct estimators, the recovered capacity ordering is stable and generally model-agnostic._
